# Domain Wall Coarsening — 3D Ising Model (Multi-Parameter Analysis)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import glob
import os

data_dir = 'data'
files = sorted(glob.glob(os.path.join(data_dir, 'coarsening_*.csv')))
datasets = {}
for f in files:
    key = os.path.basename(f).replace('.csv', '')
    datasets[key] = pd.read_csv(f)
print(f"Loaded {len(datasets)} datasets:")
for k, df in datasets.items():
    print(f"  {k}: {len(df)} rows, rho range [{df['rho'].min():.4f}, {df['rho'].max():.4f}]")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

t_quenches = [1.5, 2.0, 2.5, 3.0]
colors_t = ['#1e40af', '#3b82f6', '#10b981', '#ef4444']
t_ref = np.logspace(1, 5.3, 100)

for T, c in zip(t_quenches, colors_t):
    key = f'coarsening_N30_T{T:.2f}'
    if key in datasets:
        df = datasets[key]
        mask = df['t'] > 0
        ax2.loglog(df.loc[mask, 't'], df.loc[mask, 'rho'], 
                   alpha=0.6, color=c, label=f'T={T}', linewidth=1.2)

ax2.loglog(t_ref, 0.5 * t_ref**(-1/3), 'k--', label='z=1/3 theory', linewidth=2)
ax2.set_xlabel('t (sweeps)', fontsize=11)
ax2.set_ylabel('\u03c1 (domain wall density)', fontsize=11)
ax2.set_title('Coarsening: T_quench dependence (N=30)', fontsize=12)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.2)

# N comparison plot (T=2.5 fixed)
sizes = [20, 30, 40, 50]
colors_n = ['#7c3aed', '#3b82f6', '#10b981', '#f59e0b']

for N, c in zip(sizes, colors_n):
    key = f'coarsening_N{N}_T2.50'
    if key in datasets:
        df = datasets[key]
        mask = df['t'] > 0
        ax1.loglog(df.loc[mask, 't'], df.loc[mask, 'rho'], 
                   alpha=0.6, color=c, label=f'N={N}', linewidth=1.2)

ax1.loglog(t_ref, 0.5 * t_ref**(-1/3), 'k--', label='z=1/3 theory', linewidth=2)
ax1.set_xlabel('t (sweeps)', fontsize=11)
ax1.set_ylabel('\u03c1', fontsize=11)
ax1.set_title('Coarsening: system size dependence (T=2.5)', fontsize=12)
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig(os.path.join(data_dir, 'coarsening_verification.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
skip_frac = 0.1
rows = []

for key, df in sorted(datasets.items()):
    mask = (df['t'] > 0) & (df['rho'] > 0)
    df_fit = df[mask].copy()
    skip = int(len(df_fit) * skip_frac)
    df_fit = df_fit.iloc[skip:]
    if len(df_fit) < 10:
        continue
    log_t = np.log(df_fit['t'].values)
    log_r = np.log(df_fit['rho'].values)
    slope, intercept, r, p, se = stats.linregress(log_t, log_r)
    rows.append({
        'dataset': key,
        'z_measured': -slope,
        'z_theory': 1/3,
        'pct_error': abs(-slope - 1/3) / (1/3) * 100,
        'R2': r**2,
        'se': se
    })

df_results = pd.DataFrame(rows)
print("Coarsening exponent z (theory: 1/3 = 0.3333):")
print(df_results[['dataset', 'z_measured', 'z_theory', 'pct_error', 'R2']].to_string(index=False, float_format='{:.4f}'.format))
print(f"\nMean z = {df_results['z_measured'].mean():.4f} \u00b1 {df_results['z_measured'].std():.4f}")

## Summary

- Allen-Cahn theory predicts \u03c1(t) ~ t^(-1/3) for 3D Model A dynamics
- Coarsening exponent z \u2248 1/3 should be independent of T_quench and system size N
- Deviations at late time (large t): finite-size effects when domain size approaches N
- Deviations at early time (small t): non-universal transient before power-law sets in
- Higher T_quench (closer to Tc) may show slower effective coarsening due to critical slowing down